# QRT Challenge — Accuracy Research Notebook

## 1. Research objective and current baseline

**What.** This notebook tests a small sequence of leakage-safe improvements to the validated CatBoost sign classifier.

**How.** Every family changes one dimension, uses frozen future-facing folds, and is screened before locked confirmation.

**Why.** The validated OOF signal is real but small, while the public result suggests transportability and non-stationarity deserve scrutiny.

**Connection to the project.** Each retained change must improve classification of whether an allocation's future return is positive; the leaderboard is context only and is never a selection input.

| Validated result (not recomputed here) | Accuracy |
|---|---:|
| Official LightGBM benchmark | 52.219% |
| E10 feature/encoding configuration | 52.466% |
| CatBoost binary, standard timestamp CV | 52.631% |
| CatBoost binary, purged pooled OOF | 52.374% |
| Final threshold | 0.492 |
| Public leaderboard (context only) | 50.970% |



## 2. Setup and data loading

**What.** Load only the libraries, training data, target, and test header required for research.

**How.** Paths and seeds match the production notebook; test values and labels never enter an experiment.

**Why.** A self-contained notebook avoids hidden state while a header-only test read verifies schema without enabling test-driven choices.

**Connection to the project.** The model inputs remain the same allocation-by-timestamp panel used by the validated sign classifier.



In [ ]:
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
import itertools
import math
import random
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None


PROJECT_DIR = Path("/Users/meddeb/Documents/qrt_challenge")
DATA_PATHS = {
    "x_train": PROJECT_DIR / "X_train_9xQjqvZ.csv",
    "y_train": PROJECT_DIR / "y_train_Ppwhaz8.csv",
    "x_test": PROJECT_DIR / "X_test_1zTtEnD.csv",
}
RANDOM_STATE = 42
N_JOBS = -1
EPS = 1e-12

RUN_NATIVE_MISSING_EXPERIMENTS = False
RUN_RECENCY_EXPERIMENTS = False
RUN_HISTORICAL_ENCODING_EXPERIMENTS = False
RUN_GROUP_RELATIVE_EXPERIMENTS = False
RUN_TEMPORAL_FEATURE_EXPERIMENTS = False
RUN_FEATURE_PRUNING = False
RUN_CATBOOST_TUNING = False
RUN_MAGNITUDE_WEIGHTING = False
RUN_CONFIRMATION = False
RUN_FINAL_THRESHOLD_CALIBRATION = False

HEAVY_RUN_FLAGS = {
    "RUN_NATIVE_MISSING_EXPERIMENTS": RUN_NATIVE_MISSING_EXPERIMENTS,
    "RUN_RECENCY_EXPERIMENTS": RUN_RECENCY_EXPERIMENTS,
    "RUN_HISTORICAL_ENCODING_EXPERIMENTS": RUN_HISTORICAL_ENCODING_EXPERIMENTS,
    "RUN_GROUP_RELATIVE_EXPERIMENTS": RUN_GROUP_RELATIVE_EXPERIMENTS,
    "RUN_TEMPORAL_FEATURE_EXPERIMENTS": RUN_TEMPORAL_FEATURE_EXPERIMENTS,
    "RUN_FEATURE_PRUNING": RUN_FEATURE_PRUNING,
    "RUN_CATBOOST_TUNING": RUN_CATBOOST_TUNING,
    "RUN_MAGNITUDE_WEIGHTING": RUN_MAGNITUDE_WEIGHTING,
    "RUN_CONFIRMATION": RUN_CONFIRMATION,
    "RUN_FINAL_THRESHOLD_CALIBRATION": RUN_FINAL_THRESHOLD_CALIBRATION,
}
assert not any(HEAVY_RUN_FLAGS.values())

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

X_train_raw = pd.read_csv(DATA_PATHS["x_train"])
y_train_raw = pd.read_csv(DATA_PATHS["y_train"])
X_test_schema = pd.read_csv(DATA_PATHS["x_test"], nrows=0)

assert X_train_raw["ROW_ID"].is_unique
assert y_train_raw["ROW_ID"].is_unique
assert set(X_train_raw["ROW_ID"]) == set(y_train_raw["ROW_ID"])
train_raw = X_train_raw.merge(y_train_raw, on="ROW_ID", how="left", validate="one_to_one")
train_raw["binary_target"] = train_raw["target"].gt(0).astype("int8")
assert train_raw["target"].notna().all()
assert set(X_test_schema.columns) == set(X_train_raw.columns)

RET_COLUMNS = [f"RET_{lag}" for lag in range(1, 21)]
VOLUME_COLUMNS = [f"SIGNED_VOLUME_{lag}" for lag in range(1, 21)]
TARGET_COLUMNS = {"target", "binary_target"}
IDENTITY_COLUMNS = {"ROW_ID", "TS", "ALLOCATION", "GROUP"}




## 3. Frozen validation protocol

**What.** Freeze the existing four purged expanding timestamp folds; folds 0–2 are development and fold 3 is locked confirmation.

**How.** Every comparison uses identical validation timestamps, while development helpers reject any request containing fold 3.

**Why.** Repeated inspection of the latest fold would silently turn confirmation into another tuning set.

**Connection to the project.** Future-facing folds test whether a rule transports to later allocation returns rather than merely fitting a shuffled split.



In [ ]:
def ordered_ts_values(values):
    series = pd.Series(pd.unique(values)).dropna().astype(str)
    frame = pd.DataFrame({"TS": series})
    frame["suffix"] = pd.to_numeric(
        frame["TS"].str.extract(r"(\d+)$", expand=False), errors="coerce"
    )
    frame["has_suffix"] = frame["suffix"].notna().astype(int)
    return frame.sort_values(
        ["has_suffix", "suffix", "TS"], ascending=[False, True, True], kind="stable"
    )["TS"].to_numpy()


def timestamp_positions(values):
    values = pd.Series(values).astype(str)
    suffix = pd.to_numeric(values.str.extract(r"(\d+)$", expand=False), errors="coerce")
    if suffix.notna().all():
        return suffix.to_numpy(dtype=float)
    mapping = {ts: pos for pos, ts in enumerate(ordered_ts_values(values))}
    return values.map(mapping).to_numpy(dtype=float)


def _row_indices_for_ts(df, ts_values):
    return np.flatnonzero(df["TS"].astype(str).isin(set(map(str, ts_values))).to_numpy())


def make_purged_expanding_ts_folds(
    df, n_splits=4, min_train_fraction=0.5, validation_fraction=0.1, embargo_ts=20
):
    ordered = ordered_ts_values(df["TS"])
    n_ts = len(ordered)
    min_train = max(1, int(np.ceil(n_ts * min_train_fraction)))
    valid_size = max(1, int(np.floor(n_ts * validation_fraction)))
    first_start = min_train + embargo_ts
    last_start = n_ts - valid_size
    starts = np.unique(np.linspace(first_start, last_start, n_splits, dtype=int))
    if len(starts) != n_splits:
        raise ValueError("Insufficient timestamps for the frozen split design")
    folds = []
    for fold_id, valid_start in enumerate(starts):
        train_end = valid_start - embargo_ts
        train_ts = ordered[:train_end]
        valid_ts = ordered[valid_start:valid_start + valid_size]
        embargo_values = ordered[train_end:valid_start]
        folds.append({
            "fold": fold_id,
            "train_idx": _row_indices_for_ts(df, train_ts),
            "valid_idx": _row_indices_for_ts(df, valid_ts),
            "train_ts": train_ts,
            "valid_ts": valid_ts,
            "embargo_ts_values": embargo_values,
        })
    return folds


def validate_frozen_folds(df, folds):
    ordered = ordered_ts_values(df["TS"])
    rank = {ts: i for i, ts in enumerate(ordered)}
    records = []
    for fold in folds:
        train_idx = np.asarray(fold["train_idx"], dtype=int)
        valid_idx = np.asarray(fold["valid_idx"], dtype=int)
        train_ts = set(df.iloc[train_idx]["TS"].astype(str))
        valid_ts = set(df.iloc[valid_idx]["TS"].astype(str))
        assert not np.intersect1d(train_idx, valid_idx).size
        assert train_ts.isdisjoint(valid_ts)
        assert max(rank[ts] for ts in train_ts) < min(rank[ts] for ts in valid_ts)
        records.append({
            "fold": fold["fold"],
            "n_train_rows": len(train_idx),
            "n_valid_rows": len(valid_idx),
            "n_train_ts": len(train_ts),
            "n_valid_ts": len(valid_ts),
            "n_embargo_ts": len(fold["embargo_ts_values"]),
        })
    return pd.DataFrame(records)


PURGED_FOLDS = make_purged_expanding_ts_folds(train_raw)
DEVELOPMENT_FOLD_IDS = (0, 1, 2)
CONFIRMATION_FOLD_ID = 3
assert set(DEVELOPMENT_FOLD_IDS).isdisjoint({CONFIRMATION_FOLD_ID})
FROZEN_FOLD_REPORT = validate_frozen_folds(train_raw, PURGED_FOLDS)


def get_development_folds(requested_fold_ids=DEVELOPMENT_FOLD_IDS):
    requested = tuple(int(value) for value in requested_fold_ids)
    if CONFIRMATION_FOLD_ID in requested:
        raise PermissionError("Fold 3 is locked and cannot be used by development helpers")
    if not set(requested).issubset(DEVELOPMENT_FOLD_IDS):
        raise ValueError("Unknown development fold")
    return [deepcopy(PURGED_FOLDS[fold_id]) for fold_id in requested]


# Static protection check: development access to fold 3 must fail.
try:
    get_development_folds((0, 3))
except PermissionError:
    DEVELOPMENT_CONFIRMATION_GUARD_PASSED = True
else:
    raise AssertionError("Development helper exposed the confirmation fold")

display(FROZEN_FOLD_REPORT)




## 4. Recover the current baseline

**What.** Reconstruct the exact final feature representation, allocation encoding, preprocessing, and CatBoost configuration.

**How.** Compact builders reproduce the 53 benchmark, 24 global cross-sectional, 15 RET1-confidence, and one encoded allocation feature.

**Why.** Every candidate needs one independent baseline rather than state imported from another notebook.

**Connection to the project.** The baseline is the validated allocation sign classifier that all research candidates must beat.



In [ ]:
def _validate_predictor_frame(df):
    leaked = TARGET_COLUMNS.intersection(df.columns)
    if leaked:
        raise ValueError(f"Target columns are forbidden in feature builders: {sorted(leaked)}")
    required = {"ROW_ID", "TS", "ALLOCATION", "GROUP", *RET_COLUMNS, *VOLUME_COLUMNS}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing predictors: {sorted(missing)}")


def _safe_ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator / denominator.mask(denominator.abs() <= EPS)


def _ret_cols(horizon):
    return [f"RET_{lag}" for lag in range(1, horizon + 1)]


def _vol_cols(horizon):
    return [f"SIGNED_VOLUME_{lag}" for lag in range(1, horizon + 1)]


def build_benchmark_features(df):
    _validate_predictor_frame(df)
    work = df.copy()
    for horizon in (3, 5, 10, 15, 20):
        local_name = f"AVERAGE_PERF_{horizon}"
        work[local_name] = work[_ret_cols(horizon)].mean(axis=1)
        work[f"ALLOCATIONS_AVERAGE_PERF_{horizon}"] = (
            work.groupby("TS", dropna=False)[local_name].transform("mean")
        )
    work["STD_PERF_20"] = work[RET_COLUMNS].std(axis=1)
    work["ALLOCATIONS_STD_PERF_20"] = (
        work.groupby("TS", dropna=False)["STD_PERF_20"].transform("mean")
    )
    columns = (
        RET_COLUMNS + VOLUME_COLUMNS + ["MEDIAN_DAILY_TURNOVER"]
        + [f"AVERAGE_PERF_{h}" for h in (3, 5, 10, 15, 20)]
        + [f"ALLOCATIONS_AVERAGE_PERF_{h}" for h in (3, 5, 10, 15, 20)]
        + ["STD_PERF_20", "ALLOCATIONS_STD_PERF_20"]
    )
    return work[columns].replace([np.inf, -np.inf], np.nan)


def build_global_cross_sectional_features(df):
    _validate_predictor_frame(df)
    base = pd.DataFrame(index=df.index)
    base["RET1"] = df["RET_1"]
    base["RET_MEAN3"] = df[_ret_cols(3)].mean(axis=1)
    base["RET_MEAN5"] = df[_ret_cols(5)].mean(axis=1)
    base["RET_STD5"] = df[_ret_cols(5)].std(axis=1)
    base["RET_STD20"] = df[_ret_cols(20)].std(axis=1)
    base["SIGNED_VOLUME1"] = df["SIGNED_VOLUME_1"]
    base["SIGNED_VOLUME_MEAN5"] = df[_vol_cols(5)].mean(axis=1)
    base["TURNOVER"] = df["MEDIAN_DAILY_TURNOVER"]
    output = pd.DataFrame(index=df.index)
    for name, values in base.items():
        grouped = values.groupby(df["TS"], dropna=False)
        mean = grouped.transform("mean")
        std = grouped.transform("std")
        output[f"CS_{name}_TS_PERCENTILE"] = grouped.rank(pct=True, method="average")
        output[f"CS_{name}_TS_ZSCORE"] = _safe_ratio(values - mean, std)
        output[f"CS_{name}_MINUS_TS_MEAN"] = values - mean
    return output.replace([np.inf, -np.inf], np.nan)


def _same_sign(left, right):
    left = pd.to_numeric(left, errors="coerce")
    right = pd.to_numeric(right, errors="coerce")
    observed = left.notna() & right.notna()
    return pd.Series(
        np.where(observed, (np.sign(left) == np.sign(right)).astype(float), np.nan),
        index=left.index,
    )


def build_ret1_confidence_features(df):
    _validate_predictor_frame(df)
    ret1 = df["RET_1"]
    ret2 = df["RET_2"]
    mean3 = df[_ret_cols(3)].mean(axis=1)
    mean5 = df[_ret_cols(5)].mean(axis=1)
    mean20 = df[_ret_cols(20)].mean(axis=1)
    std5 = df[_ret_cols(5)].std(axis=1)
    std20 = df[_ret_cols(20)].std(axis=1)
    mean_abs5 = df[_ret_cols(5)].abs().mean(axis=1)
    mean_abs20 = df[_ret_cols(20)].abs().mean(axis=1)
    output = pd.DataFrame(index=df.index)
    output["R1C_ABS_RET1"] = ret1.abs()
    output["R1C_RET1_SQUARED"] = ret1.pow(2)
    output["R1C_SIGN_RET1"] = np.sign(ret1)
    output["R1C_RET1_OVER_STD5"] = _safe_ratio(ret1, std5)
    output["R1C_RET1_OVER_STD20"] = _safe_ratio(ret1, std20)
    output["R1C_RET1_MINUS_MEAN3"] = ret1 - mean3
    output["R1C_RET1_MINUS_MEAN5"] = ret1 - mean5
    output["R1C_RET1_MINUS_MEAN20"] = ret1 - mean20
    output["R1C_ABS_RET1_MINUS_MEAN5"] = (ret1 - mean5).abs()
    output["R1C_RET1_OVER_MEAN_ABS5"] = _safe_ratio(ret1, mean_abs5)
    output["R1C_RET1_OVER_MEAN_ABS20"] = _safe_ratio(ret1, mean_abs20)
    output["R1C_RET1_NEAR_ZERO"] = np.where(
        std5.notna(), ret1.abs().le(0.25 * std5).astype(float), np.nan
    )
    output["R1C_ABS_RET1_ABOVE_STD5"] = np.where(
        std5.notna(), ret1.abs().gt(std5).astype(float), np.nan
    )
    output["R1C_SIGN_AGREES_MEAN3"] = _same_sign(ret1, mean3)
    output["R1C_SIGN_AGREES_RET2"] = _same_sign(ret1, ret2)
    return output.replace([np.inf, -np.inf], np.nan)


def _finalize_feature_block(frame, source_index, name):
    if not frame.index.equals(source_index):
        raise ValueError(f"{name}: index or row order changed")
    if not frame.columns.is_unique:
        raise ValueError(f"{name}: duplicate columns")
    if TARGET_COLUMNS.intersection(frame.columns):
        raise ValueError(f"{name}: target leaked into features")
    non_numeric = frame.select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric:
        raise TypeError(f"{name}: non-numeric features: {non_numeric}")
    return frame.replace([np.inf, -np.inf], np.nan)


def build_base_feature_matrix(df, extra_blocks=(), drop_families=()):
    _validate_predictor_frame(df)
    blocks = {
        "benchmark": build_benchmark_features(df),
        "cross_sectional": build_global_cross_sectional_features(df),
        "ret1_confidence": build_ret1_confidence_features(df),
    }
    for name in extra_blocks:
        if name not in EXTRA_FEATURE_BUILDERS:
            raise KeyError(f"Unknown feature block: {name}")
        blocks[name] = EXTRA_FEATURE_BUILDERS[name](df)
    selected = {name: block for name, block in blocks.items() if name not in set(drop_families)}
    matrix = pd.concat(selected.values(), axis=1)
    matrix = _finalize_feature_block(matrix, df.index, "feature_matrix")
    family_columns = {name: list(block.columns) for name, block in selected.items()}
    metadata = {
        "row_ids": df["ROW_ID"].copy(),
        "family_columns": family_columns,
        "n_features_before_encoding": matrix.shape[1],
        "index_preserved": matrix.index.equals(df.index),
    }
    return matrix, metadata


# Builders are registered by later experiment sections without changing the baseline.
EXTRA_FEATURE_BUILDERS = {}




**What.** Define fold-safe allocation encoders and numeric preprocessing for the exact baseline and later candidates.

**How.** The baseline encoder reproduces shuffled timestamp-grouped internal cross-fitting; learned mappings and scaling statistics use outer-training rows only.

**Why.** Encoding and preprocessing must be controlled centrally so an experiment cannot introduce a second hidden change.

**Connection to the project.** Allocation history is the strongest validated identity signal, while fold-local transforms protect future-return labels.



In [ ]:
def _category_key(series):
    return pd.Series(series, index=series.index, dtype="string").fillna("__MISSING__")


class FoldSafeAllocationTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=20.0, n_inner_splits=5, random_state=42):
        self.smoothing = float(smoothing)
        self.n_inner_splits = int(n_inner_splits)
        self.random_state = int(random_state)

    def fit(self, X, y, sample_weight=None):
        y = np.asarray(y, dtype=float)
        weights = np.ones(len(y), dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        if len(X) != len(y) or len(y) != len(weights):
            raise ValueError("Target-encoding inputs are not aligned")
        work = pd.DataFrame({
            "allocation": _category_key(X["ALLOCATION"]),
            "weighted_y": weights * y,
            "weight": weights,
        }, index=X.index)
        self.global_fallback_ = float(work["weighted_y"].sum() / max(work["weight"].sum(), EPS))
        stats = work.groupby("allocation", dropna=False).agg(
            weighted_y=("weighted_y", "sum"), weight=("weight", "sum")
        )
        stats["encoded"] = (
            stats["weighted_y"] + self.smoothing * self.global_fallback_
        ) / (stats["weight"] + self.smoothing)
        self.mapping_ = stats["encoded"]
        return self

    def transform(self, X):
        values = _category_key(X["ALLOCATION"]).map(self.mapping_).fillna(self.global_fallback_)
        return pd.DataFrame({"ALLOCATION__TARGET_ENCODED": values.astype("float32")}, index=X.index)

    def fit_transform(self, X, y, groups, sample_weight=None):
        groups = pd.Series(groups, index=X.index).astype(str)
        unique_groups = pd.unique(groups)
        n_splits = min(self.n_inner_splits, len(unique_groups))
        if n_splits < 2:
            raise ValueError("Internal target encoding needs at least two timestamps")
        weights = np.ones(len(X), dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=self.random_state)
        output = pd.DataFrame(np.nan, index=X.index, columns=["ALLOCATION__TARGET_ENCODED"])
        for train_pos, valid_pos in splitter.split(unique_groups):
            train_groups = set(unique_groups[train_pos])
            valid_groups = set(unique_groups[valid_pos])
            train_mask = groups.isin(train_groups).to_numpy()
            valid_mask = groups.isin(valid_groups).to_numpy()
            inner = FoldSafeAllocationTargetEncoder(
                smoothing=self.smoothing,
                n_inner_splits=self.n_inner_splits,
                random_state=self.random_state,
            ).fit(X.loc[train_mask], np.asarray(y)[train_mask], weights[train_mask])
            output.loc[valid_mask] = inner.transform(X.loc[valid_mask]).to_numpy()
        self.fit(X, y, weights)
        if output.isna().any().any():
            raise RuntimeError("Incomplete internal target encoding")
        return output.astype("float32")


class ResearchPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_mode="linear_standard", encoding="allocation_target", encoding_params=None):
        self.numeric_mode = numeric_mode
        self.encoding = encoding
        self.encoding_params = {} if encoding_params is None else dict(encoding_params)

    def _fit_numeric(self, X):
        excluded = {"ROW_ID", "TS", "ALLOCATION", "GROUP"}
        self.numeric_columns_ = [c for c in X.select_dtypes(include=np.number).columns if c not in excluded]
        numeric = X[self.numeric_columns_].replace([np.inf, -np.inf], np.nan)
        if self.numeric_mode == "linear_standard":
            self.medians_ = numeric.median().fillna(0.0)
            filled = numeric.fillna(self.medians_)
            self.scaler_ = StandardScaler().fit(filled)
        elif self.numeric_mode == "native":
            self.medians_ = None
            self.scaler_ = None
        else:
            raise ValueError(f"Unknown numeric mode: {self.numeric_mode}")

    def _transform_numeric(self, X):
        numeric = X.reindex(columns=self.numeric_columns_).replace([np.inf, -np.inf], np.nan)
        if self.numeric_mode == "linear_standard":
            numeric = numeric.fillna(self.medians_)
            numeric = pd.DataFrame(
                self.scaler_.transform(numeric), index=X.index, columns=self.numeric_columns_
            )
        return numeric

    def _make_encoder(self):
        if self.encoding == "allocation_target":
            return FoldSafeAllocationTargetEncoder(**self.encoding_params)
        if self.encoding == "past_only_hierarchical":
            return PastOnlyHierarchicalEncoder(**self.encoding_params)
        if self.encoding == "none":
            return None
        raise ValueError(f"Unknown encoding: {self.encoding}")

    def fit_transform(self, X, y, sample_weight=None, encoder_sample_weight=None):
        self._fit_numeric(X)
        numeric = self._transform_numeric(X)
        self.encoder_ = self._make_encoder()
        parts = [numeric]
        if self.encoder_ is not None:
            encoder_weights = sample_weight if encoder_sample_weight is None else encoder_sample_weight
            if isinstance(self.encoder_, FoldSafeAllocationTargetEncoder):
                encoded = self.encoder_.fit_transform(
                    X, y, groups=X["TS"], sample_weight=encoder_weights
                )
            else:
                encoded = self.encoder_.fit_transform(X, y, sample_weight=encoder_weights)
            parts.append(encoded)
        output = pd.concat(parts, axis=1)
        self.feature_names_out_ = list(output.columns)
        return output

    def transform(self, X):
        parts = [self._transform_numeric(X)]
        if self.encoder_ is not None:
            parts.append(self.encoder_.transform(X))
        output = pd.concat(parts, axis=1)
        return output.reindex(columns=self.feature_names_out_)


BASE_CATBOOST_PARAMS = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "verbose": False,
    "random_seed": RANDOM_STATE,
    "thread_count": N_JOBS,
}

RESEARCH_BASELINE_CONFIG = {
    "experiment_id": "A0",
    "hypothesis": "Exact reconstruction of the validated production baseline",
    "feature_config": "benchmark_plus_cross_and_confidence",
    "extra_blocks": (),
    "drop_families": (),
    "encoding": "allocation_target",
    "encoding_params": {"smoothing": 20.0, "n_inner_splits": 5, "random_state": RANDOM_STATE},
    "numeric_mode": "linear_standard",
    "training_policy": "equal",
    "encoder_follows_training_weights": True,
    "model": "catboost_binary",
    "model_params": deepcopy(BASE_CATBOOST_PARAMS),
    "validation": "purged_expanding_20",
    "decision_threshold": 0.5,
    "production_threshold_reference": 0.492,
    "use_ensemble": False,
}

EXPECTED_PRODUCTION_CONTRACT = {
    "feature_config": "benchmark_plus_cross_and_confidence",
    "encoding": "allocation_target",
    "numeric_mode": "linear_standard",
    "model": "catboost_binary",
    "validation": "purged_expanding_20",
    "production_threshold_reference": 0.492,
    "use_ensemble": False,
}
assert all(RESEARCH_BASELINE_CONFIG[key] == value for key, value in EXPECTED_PRODUCTION_CONTRACT.items())

RESEARCH_RESULTS_CACHE = {}
RESEARCH_EXPERIMENT_LOG = pd.DataFrame(columns=[
    "experiment_id", "hypothesis", "stage", "development_accuracy",
    "delta_vs_baseline", "fold_deltas", "runtime_seconds", "gate_passed",
    "confirmation_run", "confirmation_delta", "final_decision",
])




## 5. Common evaluation and acceptance rules

**What.** Apply one OOF evaluation layer and conservative pre-declared gates to every candidate.

**How.** The engine reports pooled accuracy, log loss, positive rate, fold deltas, paired correctness, and a timestamp-block bootstrap interval.

**Why.** Fixed metrics and thresholds prevent acceptance rules from moving after results are visible.

**Connection to the project.** A retained model must correct future allocation signs on several time blocks, not win through one favorable timestamp cluster.



In [ ]:
DEVELOPMENT_MIN_POOLED_GAIN = 0.0008
DEVELOPMENT_MIN_NONNEGATIVE_FOLDS = 2
DEVELOPMENT_MAX_FOLD_LOSS = -0.0015
CONFIRMATION_MIN_DELTA = 0.0
FINAL_MIN_POOLED_GAIN = 0.0010
FINAL_MIN_POSITIVE_FOLDS = 3


def make_training_policy(df, train_idx, y, policy):
    train_idx = np.asarray(train_idx, dtype=int)
    selected_idx = train_idx.copy()
    weights = np.ones(len(selected_idx), dtype=float)
    positions = timestamp_positions(df.iloc[selected_idx]["TS"])
    if policy == "equal":
        pass
    elif policy in {"decay_slow", "decay_fast"}:
        unique_count = max(df.iloc[selected_idx]["TS"].nunique(), 1)
        half_life = (0.50 if policy == "decay_slow" else 0.25) * unique_count
        age = positions.max() - positions
        weights = np.power(2.0, -age / max(half_life, 1.0))
    elif policy == "recent_half":
        ordered = ordered_ts_values(df.iloc[selected_idx]["TS"])
        keep_ts = set(ordered[len(ordered) // 2:])
        keep = df.iloc[selected_idx]["TS"].astype(str).isin(keep_ts).to_numpy()
        selected_idx = selected_idx[keep]
        weights = np.ones(len(selected_idx), dtype=float)
    elif policy in {"magnitude_bottom_decile_half", "magnitude_smooth"}:
        magnitude = np.abs(np.asarray(y, dtype=float)[selected_idx])
        if policy == "magnitude_bottom_decile_half":
            cutoff = float(np.quantile(magnitude, 0.10))
            weights = np.where(magnitude <= cutoff, 0.5, 1.0)
        else:
            median = max(float(np.median(magnitude)), EPS)
            weights = np.clip(0.5 + magnitude / median, 0.5, 2.0)
    else:
        raise ValueError(f"Unknown training policy: {policy}")
    weights = weights / max(float(weights.mean()), EPS)
    return selected_idx, weights


def _build_model_frame_core(predictors, config):
    matrix, metadata = build_base_feature_matrix(
        predictors,
        extra_blocks=tuple(config.get("extra_blocks", ())),
        drop_families=tuple(config.get("drop_families", ())),
    )
    frame = matrix.copy()
    for column in ("ROW_ID", "TS", "ALLOCATION", "GROUP"):
        frame[column] = predictors[column].to_numpy()
    if TARGET_COLUMNS.intersection(frame.columns):
        raise AssertionError("Target column entered the model frame")
    assert frame.index.equals(predictors.index)
    assert frame["ROW_ID"].equals(predictors["ROW_ID"])
    return frame, metadata


def make_estimator(config):
    if CatBoostClassifier is None:
        raise ImportError("CatBoost is required only when an explicit RUN flag is enabled")
    return CatBoostClassifier(**deepcopy(config["model_params"]))


def _run_fold_set(experiment_id, config, df, folds, stage):
    predictors = df.drop(columns=["target", "binary_target"])
    X_model, feature_metadata = build_model_frame(predictors, config)
    y_binary = df["binary_target"].to_numpy(dtype=int)
    y_continuous = df["target"].to_numpy(dtype=float)
    oof_probability = np.full(len(df), np.nan, dtype=float)
    oof_fold = np.full(len(df), -1, dtype=int)
    fold_records = []
    fitted = []
    started = time.perf_counter()
    for fold in folds:
        fold_id = int(fold["fold"])
        train_idx, model_weights = make_training_policy(
            df, fold["train_idx"], y_continuous, config["training_policy"]
        )
        valid_idx = np.asarray(fold["valid_idx"], dtype=int)
        preprocessor = ResearchPreprocessor(
            numeric_mode=config["numeric_mode"],
            encoding=config["encoding"],
            encoding_params=config.get("encoding_params", {}),
        )
        encoder_weights = model_weights if config.get("encoder_follows_training_weights", True) else np.ones_like(model_weights)
        X_train = preprocessor.fit_transform(
            X_model.iloc[train_idx], y_binary[train_idx],
            sample_weight=model_weights, encoder_sample_weight=encoder_weights,
        )
        X_valid = preprocessor.transform(X_model.iloc[valid_idx])
        if TARGET_COLUMNS.intersection(X_train.columns) or TARGET_COLUMNS.intersection(X_valid.columns):
            raise AssertionError("Target leaked after preprocessing")
        estimator = make_estimator(config)
        estimator.fit(X_train, y_binary[train_idx], sample_weight=model_weights)
        probability = np.asarray(estimator.predict_proba(X_valid)[:, 1], dtype=float)
        prediction = (probability >= 0.5).astype(int)
        oof_probability[valid_idx] = probability
        oof_fold[valid_idx] = fold_id
        fold_records.append({
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
            "accuracy": float(accuracy_score(y_binary[valid_idx], prediction)),
            "log_loss": float(log_loss(y_binary[valid_idx], probability, labels=[0, 1])),
            "predicted_positive_rate": float(prediction.mean()),
        })
        fitted.append({
            "fold": fold_id,
            "estimator": estimator,
            "preprocessor": preprocessor,
            "train_idx": train_idx,
            "valid_idx": valid_idx,
        })
    covered = np.isfinite(oof_probability) & (oof_fold >= 0)
    binary = np.full(len(df), -1, dtype=int)
    binary[covered] = (oof_probability[covered] >= 0.5).astype(int)
    result = {
        "experiment_id": experiment_id,
        "hypothesis": config["hypothesis"],
        "stage": stage,
        "config": deepcopy(config),
        "oof_probability": oof_probability,
        "oof_binary": binary,
        "fold_ids": oof_fold,
        "fold_metrics": pd.DataFrame(fold_records),
        "pooled_accuracy": float(accuracy_score(y_binary[covered], binary[covered])),
        "pooled_log_loss": float(log_loss(y_binary[covered], oof_probability[covered], labels=[0, 1])),
        "predicted_positive_rate": float(binary[covered].mean()),
        "runtime_seconds": float(time.perf_counter() - started),
        "feature_metadata": feature_metadata,
        "fitted_folds": fitted,
        "covered_mask": covered,
    }
    return result


def paired_timestamp_bootstrap(df, y_true, candidate_pred, baseline_pred, mask, n_bootstrap=2000, random_state=42):
    positions = np.flatnonzero(np.asarray(mask, dtype=bool))
    work = pd.DataFrame({
        "TS": df.iloc[positions]["TS"].astype(str).to_numpy(),
        "candidate_correct": (np.asarray(candidate_pred)[positions] == np.asarray(y_true)[positions]).astype(int),
        "baseline_correct": (np.asarray(baseline_pred)[positions] == np.asarray(y_true)[positions]).astype(int),
    })
    by_ts = work.groupby("TS").agg(
        candidate_correct=("candidate_correct", "sum"),
        baseline_correct=("baseline_correct", "sum"),
        n=("candidate_correct", "size"),
    )
    observed = float(
        (by_ts["candidate_correct"].sum() - by_ts["baseline_correct"].sum()) / by_ts["n"].sum()
    )
    rng = np.random.default_rng(random_state)
    samples = np.empty(int(n_bootstrap), dtype=float)
    for i in range(int(n_bootstrap)):
        sampled = by_ts.iloc[rng.integers(0, len(by_ts), len(by_ts))]
        samples[i] = (
            sampled["candidate_correct"].sum() - sampled["baseline_correct"].sum()
        ) / sampled["n"].sum()
    return {
        "observed_delta": observed,
        "confidence_interval_95": tuple(np.quantile(samples, [0.025, 0.975]).astype(float)),
        "n_timestamps": int(len(by_ts)),
    }


def summarize_candidate_vs_baseline(candidate, baseline, df):
    common = candidate["covered_mask"] & baseline["covered_mask"]
    if not np.array_equal(candidate["covered_mask"], baseline["covered_mask"]):
        raise ValueError("Candidate and baseline do not cover identical validation rows")
    y = df["binary_target"].to_numpy(dtype=int)
    fold_rows = []
    for fold_id in sorted(np.unique(candidate["fold_ids"][common])):
        mask = common & (candidate["fold_ids"] == fold_id)
        candidate_accuracy = accuracy_score(y[mask], candidate["oof_binary"][mask])
        baseline_accuracy = accuracy_score(y[mask], baseline["oof_binary"][mask])
        fold_rows.append({
            "fold": int(fold_id),
            "candidate_accuracy": float(candidate_accuracy),
            "baseline_accuracy": float(baseline_accuracy),
            "delta": float(candidate_accuracy - baseline_accuracy),
        })
    fold_deltas = pd.DataFrame(fold_rows)
    pooled_delta = float(candidate["pooled_accuracy"] - baseline["pooled_accuracy"])
    bootstrap = paired_timestamp_bootstrap(
        df, y, candidate["oof_binary"], baseline["oof_binary"], common
    )
    stage = candidate["stage"]
    if stage == "development":
        gate_passed = bool(
            pooled_delta >= DEVELOPMENT_MIN_POOLED_GAIN
            and int((fold_deltas["delta"] >= 0).sum()) >= DEVELOPMENT_MIN_NONNEGATIVE_FOLDS
            and float(fold_deltas["delta"].min()) >= DEVELOPMENT_MAX_FOLD_LOSS
        )
    else:
        gate_passed = None
    return {
        "candidate_accuracy": candidate["pooled_accuracy"],
        "baseline_accuracy": baseline["pooled_accuracy"],
        "delta_vs_baseline": pooled_delta,
        "fold_deltas": fold_deltas,
        "worst_fold_accuracy": float(candidate["fold_metrics"]["accuracy"].min()),
        "predicted_positive_rate": candidate["predicted_positive_rate"],
        "log_loss": candidate["pooled_log_loss"],
        "paired_correctness_difference": bootstrap["observed_delta"],
        "paired_timestamp_bootstrap": bootstrap,
        "gate_passed": gate_passed,
    }


def _upsert_research_log(
    result, comparison=None, decision="DEVELOPMENT_ONLY", gate_override=None
):
    global RESEARCH_EXPERIMENT_LOG
    fold_deltas = None if comparison is None else comparison["fold_deltas"]["delta"].round(6).tolist()
    row = {
        "experiment_id": result["experiment_id"],
        "hypothesis": result["hypothesis"],
        "stage": result["stage"],
        "development_accuracy": result["pooled_accuracy"] if result["stage"] == "development" else np.nan,
        "delta_vs_baseline": np.nan if comparison is None else comparison["delta_vs_baseline"],
        "fold_deltas": fold_deltas,
        "runtime_seconds": result["runtime_seconds"],
        "gate_passed": (
            gate_override
            if gate_override is not None
            else (None if comparison is None else comparison["gate_passed"])
        ),
        "confirmation_run": result["stage"] == "confirmation",
        "confirmation_delta": (
            comparison["delta_vs_baseline"] if result["stage"] == "confirmation" and comparison is not None else np.nan
        ),
        "final_decision": decision,
    }
    RESEARCH_EXPERIMENT_LOG = RESEARCH_EXPERIMENT_LOG.loc[
        ~((RESEARCH_EXPERIMENT_LOG["experiment_id"] == result["experiment_id"])
          & (RESEARCH_EXPERIMENT_LOG["stage"] == result["stage"]))
    ]
    RESEARCH_EXPERIMENT_LOG = pd.concat(
        [RESEARCH_EXPERIMENT_LOG, pd.DataFrame([row])], ignore_index=True
    )


def ensure_development_baseline(df=train_raw):
    cache_key = "A0::development"
    if cache_key not in RESEARCH_RESULTS_CACHE:
        result = _run_fold_set(
            "A0", RESEARCH_BASELINE_CONFIG, df, get_development_folds(), "development"
        )
        RESEARCH_RESULTS_CACHE[cache_key] = result
        _upsert_research_log(result, comparison=None, decision="BASELINE")
    return RESEARCH_RESULTS_CACHE[cache_key]


def run_dev_experiment(experiment_id, config, df=train_raw):
    if experiment_id == "A0":
        return ensure_development_baseline(df)
    baseline = ensure_development_baseline(df)
    result = _run_fold_set(experiment_id, config, df, get_development_folds(), "development")
    comparison = summarize_candidate_vs_baseline(result, baseline, df)
    result["comparison"] = comparison
    RESEARCH_RESULTS_CACHE[f"{experiment_id}::development"] = result
    _upsert_research_log(
        result, comparison, decision="PASS_DEV" if comparison["gate_passed"] else "REJECT_DEV"
    )
    return result


def run_dev_family(registry, experiment_ids, df=train_raw):
    outputs = {}
    for experiment_id in experiment_ids:
        outputs[experiment_id] = run_dev_experiment(experiment_id, registry[experiment_id], df)
    return outputs




## 6. Experiment A — Native missing values

**What.** Compare the current imputed/scaled baseline with native NaNs, then add only five compact availability signals.

**How.** A1 changes preprocessing alone; A2 adds missing flags/counts; optional A3 removes the sparse raw `SIGNED_VOLUME_1` values but retains availability.

**Why.** `SIGNED_VOLUME_1` is missing on about 73% of training rows, and median imputation makes missing indistinguishable from typical.

**Connection to the project.** Availability may identify regimes where recent volume is less reliable when predicting the future sign, without recreating the rejected return-volume interaction block.



In [ ]:
def _leading_missing_streak(frame):
    missing = frame.isna().to_numpy(dtype=bool)
    streak = np.zeros(len(frame), dtype=int)
    active = np.ones(len(frame), dtype=bool)
    for column_pos in range(missing.shape[1]):
        active &= missing[:, column_pos]
        streak += active.astype(int)
    return streak


def build_missingness_features(df):
    _validate_predictor_frame(df)
    output = pd.DataFrame(index=df.index)
    output["MISS_SIGNED_VOLUME_1"] = df["SIGNED_VOLUME_1"].isna().astype("int8")
    output["MISS_TURNOVER"] = df["MEDIAN_DAILY_TURNOVER"].isna().astype("int8")
    output["VOLUME_AVAILABLE_COUNT_5"] = df[_vol_cols(5)].notna().sum(axis=1).astype("int8")
    output["VOLUME_AVAILABLE_COUNT_20"] = df[VOLUME_COLUMNS].notna().sum(axis=1).astype("int8")
    output["VOLUME_LEADING_MISSING_STREAK"] = _leading_missing_streak(df[_vol_cols(20)])
    return _finalize_feature_block(output, df.index, "missingness")


EXTRA_FEATURE_BUILDERS["missingness"] = build_missingness_features

A_REGISTRY = {
    "A0": deepcopy(RESEARCH_BASELINE_CONFIG),
    "A1": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "A1",
        "hypothesis": "Native CatBoost NaNs outperform median imputation and scaling",
        "numeric_mode": "native",
    },
    "A2": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "A2",
        "hypothesis": "Compact missingness signals add stable information under native NaN handling",
        "numeric_mode": "native",
        "extra_blocks": ("missingness",),
    },
    "A3": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "A3",
        "hypothesis": "Availability is more robust than the extremely sparse raw latest volume",
        "numeric_mode": "native",
        "extra_blocks": ("missingness",),
        "drop_columns": (
            "SIGNED_VOLUME_1",
            "CS_SIGNED_VOLUME1_TS_PERCENTILE",
            "CS_SIGNED_VOLUME1_TS_ZSCORE",
            "CS_SIGNED_VOLUME1_MINUS_TS_MEAN",
        ),
    },
}


def build_model_frame(predictors, config):
    """Build the model frame once, then apply a candidate's explicit column drops."""
    frame, metadata = _build_model_frame_core(predictors, config)
    drop_columns = [column for column in config.get("drop_columns", ()) if column in frame.columns]
    if drop_columns:
        frame = frame.drop(columns=drop_columns)
        metadata = deepcopy(metadata)
        metadata["explicitly_dropped_columns"] = drop_columns
    assert frame.index.equals(predictors.index)
    assert frame["ROW_ID"].equals(predictors["ROW_ID"])
    assert not TARGET_COLUMNS.intersection(frame.columns)
    return frame, metadata


if RUN_NATIVE_MISSING_EXPERIMENTS:
    A_RESULTS = run_dev_family(A_REGISTRY, ["A1", "A2", "A3"])
else:
    A_RESULTS = {}
    print("Experiment A is prepared; RUN_NATIVE_MISSING_EXPERIMENTS=False.")




## 7. Experiment B — Recency weighting

**What.** Test equal history against two fixed exponential decays and a recent-half timestamp window.

**How.** Timestamp age is computed inside each outer-training fold; recency weights are also used in allocation target statistics so model and encoding represent the same history.

**Why.** Older data reduce variance but may encode stale allocation behavior; fixed rules answer this without tuning a continuous half-life.

**Connection to the project.** The experiment asks which historical allocation returns are most relevant to signs in the next unseen time block.



In [ ]:
B_REGISTRY = {
    "B0": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "B0",
        "hypothesis": "Equal-weight history control; identical to A0",
    },
    "B1": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "B1",
        "hypothesis": "Slow exponential recency weighting reduces mild historical staleness",
        "training_policy": "decay_slow",
        "encoder_follows_training_weights": True,
    },
    "B2": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "B2",
        "hypothesis": "Faster exponential recency weighting helps under stronger drift",
        "training_policy": "decay_fast",
        "encoder_follows_training_weights": True,
    },
    "B3": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "B3",
        "hypothesis": "The recent half of timestamps is more relevant than the full expanding history",
        "training_policy": "recent_half",
        "encoder_follows_training_weights": True,
    },
}

if RUN_RECENCY_EXPERIMENTS:
    B_RESULTS = run_dev_family(B_REGISTRY, ["B1", "B2", "B3"])
else:
    B_RESULTS = {}
    print("Experiment B is prepared; RUN_RECENCY_EXPERIMENTS=False.")




## 8. Experiment C — Past-only hierarchical allocation encoding

**What.** Replace shuffled allocation encoding with allocation → group → global history that is strictly chronological.

**How.** Encodings are emitted for every timestamp before any labels from that timestamp update weighted counts; two smoothing settings and one fixed decay are pre-specified.

**Why.** The outer-fold-safe baseline remains less deployment-like inside training because its internal timestamp folds are shuffled.

**Connection to the project.** The encoder estimates an allocation's prior positive-return tendency using only information that would have existed before the predicted timestamp.



In [ ]:
def _empty_history_state():
    return {
        "global_weight": 0.0,
        "global_positive": 0.0,
        "group_weight": {},
        "group_positive": {},
        "allocation_weight": {},
        "allocation_positive": {},
        "allocation_group": {},
        "last_position": None,
    }


def _decay_history_state(state, factor):
    state["global_weight"] *= factor
    state["global_positive"] *= factor
    for name in ("group_weight", "group_positive", "allocation_weight", "allocation_positive"):
        for key in list(state[name]):
            state[name][key] *= factor


def _hierarchical_rate(state, allocation, group, allocation_smoothing, group_smoothing):
    global_weight = state["global_weight"]
    global_rate = (
        state["global_positive"] / global_weight if global_weight > EPS else 0.5
    )
    group_weight = state["group_weight"].get(group, 0.0)
    group_positive = state["group_positive"].get(group, 0.0)
    group_rate = (
        group_positive + group_smoothing * global_rate
    ) / (group_weight + group_smoothing)
    allocation_weight = state["allocation_weight"].get(allocation, 0.0)
    allocation_positive = state["allocation_positive"].get(allocation, 0.0)
    allocation_rate = (
        allocation_positive + allocation_smoothing * group_rate
    ) / (allocation_weight + allocation_smoothing)
    return float(allocation_rate), float(allocation_weight)


def past_only_hierarchical_core(
    timestamps,
    positions,
    allocations,
    groups,
    targets=None,
    sample_weight=None,
    allocation_smoothing=20.0,
    group_smoothing=100.0,
    decay_half_life=None,
    initial_state=None,
    update=True,
):
    """Pure chronological core. Every timestamp is encoded before it can update history."""
    n_rows = len(timestamps)
    if not (len(positions) == len(allocations) == len(groups) == n_rows):
        raise ValueError("Historical encoding inputs are not aligned")
    if update and (targets is None or len(targets) != n_rows):
        raise ValueError("Targets are required when history is updated")
    weights = [1.0] * n_rows if sample_weight is None else [float(v) for v in sample_weight]
    if len(weights) != n_rows:
        raise ValueError("Historical sample weights are not aligned")
    state = deepcopy(_empty_history_state() if initial_state is None else initial_state)
    rates = [None] * n_rows
    counts = [None] * n_rows
    order = sorted(range(n_rows), key=lambda i: (float(positions[i]), str(timestamps[i]), i))
    cursor = 0
    while cursor < n_rows:
        first = order[cursor]
        timestamp = str(timestamps[first])
        position = float(positions[first])
        end = cursor
        batch = []
        while end < n_rows and str(timestamps[order[end]]) == timestamp:
            batch.append(order[end])
            end += 1
        if state["last_position"] is not None and decay_half_life is not None:
            gap = max(0.0, position - float(state["last_position"]))
            _decay_history_state(state, math.pow(0.5, gap / float(decay_half_life)))
        state["last_position"] = position
        for row in batch:
            rate, count = _hierarchical_rate(
                state, str(allocations[row]), str(groups[row]),
                float(allocation_smoothing), float(group_smoothing),
            )
            rates[row] = rate
            counts[row] = count
        if update:
            for row in batch:
                allocation = str(allocations[row])
                group = str(groups[row])
                weight = float(weights[row])
                target = float(targets[row])
                state["global_weight"] += weight
                state["global_positive"] += weight * target
                state["group_weight"][group] = state["group_weight"].get(group, 0.0) + weight
                state["group_positive"][group] = state["group_positive"].get(group, 0.0) + weight * target
                state["allocation_weight"][allocation] = state["allocation_weight"].get(allocation, 0.0) + weight
                state["allocation_positive"][allocation] = state["allocation_positive"].get(allocation, 0.0) + weight * target
                state["allocation_group"][allocation] = group
        cursor = end
    return rates, counts, state


class PastOnlyHierarchicalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, allocation_smoothing=20.0, group_smoothing=100.0, decay_half_life=None):
        self.allocation_smoothing = float(allocation_smoothing)
        self.group_smoothing = float(group_smoothing)
        self.decay_half_life = None if decay_half_life is None else float(decay_half_life)

    def _core(self, X, y=None, sample_weight=None, initial_state=None, update=True):
        return past_only_hierarchical_core(
            timestamps=X["TS"].astype(str).tolist(),
            positions=timestamp_positions(X["TS"]).tolist(),
            allocations=_category_key(X["ALLOCATION"]).astype(str).tolist(),
            groups=_category_key(X["GROUP"]).astype(str).tolist(),
            targets=None if y is None else np.asarray(y, dtype=float).tolist(),
            sample_weight=None if sample_weight is None else np.asarray(sample_weight, dtype=float).tolist(),
            allocation_smoothing=self.allocation_smoothing,
            group_smoothing=self.group_smoothing,
            decay_half_life=self.decay_half_life,
            initial_state=initial_state,
            update=update,
        )

    @staticmethod
    def _frame(index, rates, counts):
        return pd.DataFrame({
            "ALLOCATION__PAST_HIER_RATE": np.asarray(rates, dtype="float32"),
            "ALLOCATION__PAST_LOG_COUNT": np.log1p(np.asarray(counts, dtype=float)).astype("float32"),
        }, index=index)

    def fit(self, X, y, sample_weight=None):
        _, _, self.state_ = self._core(X, y, sample_weight, initial_state=None, update=True)
        return self

    def fit_transform(self, X, y, sample_weight=None):
        rates, counts, self.state_ = self._core(X, y, sample_weight, initial_state=None, update=True)
        return self._frame(X.index, rates, counts)

    def transform(self, X):
        rates, counts, _ = self._core(
            X, y=None, sample_weight=None, initial_state=self.state_, update=False
        )
        return self._frame(X.index, rates, counts)


def run_historical_encoder_unit_tests():
    base = pd.DataFrame({
        "TS": ["DATE_0001", "DATE_0001", "DATE_0002", "DATE_0003"],
        "ALLOCATION": ["A", "B", "A", "A"],
        "GROUP": ["G1", "G1", "G1", "G1"],
    })
    y_first = np.array([0, 1, 1, 0], dtype=int)
    y_same_timestamp_changed = np.array([1, 1, 1, 0], dtype=int)
    y_future_changed = np.array([0, 1, 0, 1], dtype=int)
    encoder = PastOnlyHierarchicalEncoder(20.0, 100.0)
    encoded_first = encoder.fit_transform(base, y_first)
    encoded_same = PastOnlyHierarchicalEncoder(20.0, 100.0).fit_transform(
        base, y_same_timestamp_changed
    )
    encoded_future = PastOnlyHierarchicalEncoder(20.0, 100.0).fit_transform(
        base, y_future_changed
    )
    assert np.allclose(encoded_first.iloc[:2], encoded_same.iloc[:2])
    assert np.allclose(encoded_first.iloc[:2], encoded_future.iloc[:2])
    assert np.allclose(encoded_first.iloc[2], encoded_future.iloc[2])
    fitted = PastOnlyHierarchicalEncoder(20.0, 100.0).fit(base.iloc[:3], y_first[:3])
    unseen = pd.DataFrame({"TS": ["DATE_0004"], "ALLOCATION": ["UNSEEN"], "GROUP": ["G1"]})
    unseen_value = float(fitted.transform(unseen)["ALLOCATION__PAST_HIER_RATE"].iloc[0])
    assert 0.0 <= unseen_value <= 1.0
    return {
        "future_targets_do_not_change_earlier_rows": True,
        "same_timestamp_labels_are_isolated": True,
        "unseen_allocation_fallback_is_finite": True,
    }


HISTORICAL_ENCODER_TESTS = run_historical_encoder_unit_tests()

C_REGISTRY = {
    "C1": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "C1",
        "hypothesis": "Past-only allocation history with moderate hierarchical shrinkage transports better",
        "encoding": "past_only_hierarchical",
        "encoding_params": {"allocation_smoothing": 20.0, "group_smoothing": 100.0},
    },
    "C2": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "C2",
        "hypothesis": "Stronger allocation and group shrinkage improves stability",
        "encoding": "past_only_hierarchical",
        "encoding_params": {"allocation_smoothing": 100.0, "group_smoothing": 500.0},
    },
    "C3": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "C3",
        "hypothesis": "A fixed 504-timestamp decay makes allocation history more deployment-relevant",
        "encoding": "past_only_hierarchical",
        "encoding_params": {
            "allocation_smoothing": 20.0,
            "group_smoothing": 100.0,
            "decay_half_life": 504.0,
        },
    },
}

if RUN_HISTORICAL_ENCODING_EXPERIMENTS:
    C_RESULTS = run_dev_family(C_REGISTRY, ["C1", "C2", "C3"])
else:
    C_RESULTS = {}
    print("Experiment C is prepared; RUN_HISTORICAL_ENCODING_EXPERIMENTS=False.")




## 9. Experiment D — Group-relative robust cross-sectional features

**What.** Add only within-`TS × GROUP` ranks, median deviations, MAD z-scores, and four compact group context measures.

**How.** Four economically interpretable bases are used; cells with fewer than 20 observations are set to missing rather than trusted.

**Why.** Existing features describe global timestamp position but can hide relative behavior inside the four allocation groups.

**Connection to the project.** Group-relative momentum and dispersion may distinguish allocations whose recent return is unusual among comparable allocations.



In [ ]:
MIN_GROUP_TIMESTAMP_OBSERVATIONS = 20


def _mad(values):
    values = pd.to_numeric(values, errors="coerce")
    median = values.median()
    return (values - median).abs().median()


def build_group_relative_features(df):
    _validate_predictor_frame(df)
    keys = [df["TS"], df["GROUP"]]
    short = df[_ret_cols(3)].mean(axis=1)
    long = df[[f"RET_{lag}" for lag in range(6, 21)]].mean(axis=1)
    bases = {
        "RET1": df["RET_1"],
        "RET_MEAN3": short,
        "RET_MEAN5": df[_ret_cols(5)].mean(axis=1),
        "RET_STD20": df[_ret_cols(20)].std(axis=1),
    }
    output = pd.DataFrame(index=df.index)
    cell_count = df["RET_1"].groupby(keys, dropna=False).transform("count")
    reliable = cell_count >= MIN_GROUP_TIMESTAMP_OBSERVATIONS
    rank_cache = {}
    for name, values in bases.items():
        grouped = values.groupby(keys, dropna=False)
        median = grouped.transform("median")
        mad = grouped.transform(_mad)
        rank = grouped.rank(pct=True, method="average")
        rank_cache[name] = rank
        output[f"GR_{name}_PERCENTILE"] = rank.where(reliable)
        output[f"GR_{name}_MINUS_MEDIAN"] = (values - median).where(reliable)
        output[f"GR_{name}_ROBUST_Z"] = _safe_ratio(values - median, 1.4826 * mad).where(reliable)
    ret1_group = df["RET_1"].groupby(keys, dropna=False)
    output["GR_RET1_POSITIVE_SHARE"] = df["RET_1"].gt(0).groupby(keys, dropna=False).transform("mean").where(reliable)
    output["GR_RET1_MAD"] = ret1_group.transform(_mad).where(reliable)
    output["GR_MEDIAN_SHORT_MINUS_LONG"] = (short - long).groupby(keys, dropna=False).transform("median").where(reliable)
    long_rank = long.groupby(keys, dropna=False).rank(pct=True, method="average")
    output["GR_RELATIVE_MOMENTUM_RANK_SPREAD"] = (rank_cache["RET_MEAN3"] - long_rank).where(reliable)
    return _finalize_feature_block(output, df.index, "group_relative")


EXTRA_FEATURE_BUILDERS["group_relative"] = build_group_relative_features

D_REGISTRY = {
    "D1": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "D1",
        "hypothesis": "Robust within-group cross-sectional positions add information beyond global ranks",
        "extra_blocks": ("group_relative",),
    },
}

if RUN_GROUP_RELATIVE_EXPERIMENTS:
    D_RESULTS = run_dev_family(D_REGISTRY, ["D1"])
else:
    D_RESULTS = {}
    print("Experiment D is prepared; RUN_GROUP_RELATIVE_EXPERIMENTS=False.")




## 10. Experiment E — Compact temporal summaries

**What.** Summarize the 20 observed return lags with ten decay, volatility, momentum, persistence, and autocorrelation features.

**How.** All calculations stay within a row and give the greatest weight to `RET_1`; no future timestamp or target is used.

**Why.** Raw lags and arithmetic averages may not expose decay, acceleration, or persistence efficiently to shallow trees.

**Connection to the project.** These features describe how an allocation arrived at its latest state before its future return sign is classified.



In [ ]:
def _weighted_row_moments(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.broadcast_to(np.asarray(weights, dtype=float), values.shape)
    valid = np.isfinite(values)
    effective = np.where(valid, weights, 0.0)
    denominator = effective.sum(axis=1)
    mean = np.divide(
        np.where(valid, values * weights, 0.0).sum(axis=1),
        denominator,
        out=np.full(len(values), np.nan),
        where=denominator > EPS,
    )
    variance = np.divide(
        np.where(valid, weights * (values - mean[:, None]) ** 2, 0.0).sum(axis=1),
        denominator,
        out=np.full(len(values), np.nan),
        where=denominator > EPS,
    )
    return mean, np.sqrt(variance)


def _row_lag1_autocorrelation(values, min_pairs=8):
    left = np.asarray(values[:, :-1], dtype=float)
    right = np.asarray(values[:, 1:], dtype=float)
    valid = np.isfinite(left) & np.isfinite(right)
    count = valid.sum(axis=1)
    safe = np.maximum(count, 1)
    left_mean = np.where(valid, left, 0.0).sum(axis=1) / safe
    right_mean = np.where(valid, right, 0.0).sum(axis=1) / safe
    left_centered = np.where(valid, left - left_mean[:, None], 0.0)
    right_centered = np.where(valid, right - right_mean[:, None], 0.0)
    denominator = np.sqrt((left_centered ** 2).sum(axis=1) * (right_centered ** 2).sum(axis=1))
    output = np.full(len(values), np.nan)
    usable = (count >= min_pairs) & (denominator > EPS)
    output[usable] = (left_centered * right_centered).sum(axis=1)[usable] / denominator[usable]
    return output


def build_temporal_summary_features(df):
    _validate_predictor_frame(df)
    values = df[RET_COLUMNS].to_numpy(dtype=float)
    lags = np.arange(20, dtype=float)
    output = pd.DataFrame(index=df.index)
    for half_life in (2, 5, 10):
        weights = np.power(0.5, lags / half_life)
        mean, std = _weighted_row_moments(values, weights)
        output[f"TEMP_EWM_RETURN_HL{half_life}"] = mean
        if half_life in (5, 10):
            output[f"TEMP_EWM_VOL_HL{half_life}"] = std
    short = df[_ret_cols(3)].mean(axis=1)
    output["TEMP_SHORT_MINUS_LONG"] = short - df[[f"RET_{lag}" for lag in range(6, 21)]].mean(axis=1)
    output["TEMP_MOMENTUM_ACCELERATION"] = short - df[["RET_4", "RET_5", "RET_6"]].mean(axis=1)
    output["TEMP_POSITIVE_SHARE_5"] = df[_ret_cols(5)].gt(0).where(df[_ret_cols(5)].notna()).mean(axis=1)
    output["TEMP_POSITIVE_SHARE_10"] = df[_ret_cols(10)].gt(0).where(df[_ret_cols(10)].notna()).mean(axis=1)
    output["TEMP_LAG1_AUTOCORRELATION"] = _row_lag1_autocorrelation(values, min_pairs=8)
    return _finalize_feature_block(output, df.index, "temporal_summary")


EXTRA_FEATURE_BUILDERS["temporal_summary"] = build_temporal_summary_features

E_REGISTRY = {
    "E1": {
        **deepcopy(RESEARCH_BASELINE_CONFIG),
        "experiment_id": "E1",
        "hypothesis": "Compact decayed temporal summaries expose stable momentum dynamics",
        "extra_blocks": ("temporal_summary",),
    },
}

if RUN_TEMPORAL_FEATURE_EXPERIMENTS:
    E_RESULTS = run_dev_family(E_REGISTRY, ["E1"])
else:
    E_RESULTS = {}
    print("Experiment E is prepared; RUN_TEMPORAL_FEATURE_EXPERIMENTS=False.")




## 11. Experiment F — Feature-family pruning

**What.** Diagnose whole feature families by OOF permutation, then allow at most three leave-one-family-out retraining tests.

**How.** A confirmed A–E representation must be frozen first; validation values are permuted within timestamp where possible, and SHAP is interpretation-only.

**Why.** Removing a weak family can lower variance, but selecting many individual columns would overfit the same folds.

**Connection to the project.** The final sign classifier should retain only return, volume, context, and history families that contribute on unseen timestamps.



In [ ]:
CONFIRMED_EXPERIMENT_CONFIGS = {}
FROZEN_REPRESENTATION_EXPERIMENT_ID = None
SELECTED_PRUNING_FAMILIES = []


def get_frozen_representation_config():
    if FROZEN_REPRESENTATION_EXPERIMENT_ID is None:
        raise RuntimeError("Confirm one A–E representation before pruning or tuning")
    if FROZEN_REPRESENTATION_EXPERIMENT_ID not in CONFIRMED_EXPERIMENT_CONFIGS:
        raise RuntimeError("The selected representation has not passed locked confirmation")
    return deepcopy(CONFIRMED_EXPERIMENT_CONFIGS[FROZEN_REPRESENTATION_EXPERIMENT_ID])


def _permute_columns_within_timestamp(frame, columns, timestamps, rng):
    output = frame.copy()
    columns = [column for column in columns if column in output.columns]
    if not columns:
        return output
    ts_values = pd.Series(timestamps, index=output.index).astype(str)
    for _, row_index in ts_values.groupby(ts_values).groups.items():
        row_index = list(row_index)
        if len(row_index) > 1:
            permuted_index = rng.permutation(row_index)
            output.loc[row_index, columns] = output.loc[permuted_index, columns].to_numpy()
    return output


def oof_family_permutation_diagnostic(result, df=train_raw, random_state=RANDOM_STATE):
    if result.get("stage") != "development":
        raise ValueError("Family permutation is restricted to development OOF models")
    config = result["config"]
    predictors = df.drop(columns=["target", "binary_target"])
    X_model, metadata = build_model_frame(predictors, config)
    y = df["binary_target"].to_numpy(dtype=int)
    records = []
    rng = np.random.default_rng(random_state)
    for fitted in result["fitted_folds"]:
        valid_idx = np.asarray(fitted["valid_idx"], dtype=int)
        processed = fitted["preprocessor"].transform(X_model.iloc[valid_idx])
        baseline_probability = np.asarray(
            fitted["estimator"].predict_proba(processed)[:, 1], dtype=float
        )
        baseline_accuracy = accuracy_score(y[valid_idx], baseline_probability >= 0.5)
        timestamps = X_model.iloc[valid_idx]["TS"]
        for family, raw_columns in metadata["family_columns"].items():
            candidate_columns = [column for column in raw_columns if column in processed.columns]
            if not candidate_columns:
                continue
            permuted = _permute_columns_within_timestamp(
                processed, candidate_columns, timestamps, rng
            )
            probability = np.asarray(
                fitted["estimator"].predict_proba(permuted)[:, 1], dtype=float
            )
            permuted_accuracy = accuracy_score(y[valid_idx], probability >= 0.5)
            records.append({
                "fold": fitted["fold"],
                "family": family,
                "baseline_accuracy": float(baseline_accuracy),
                "permuted_accuracy": float(permuted_accuracy),
                "accuracy_drop": float(baseline_accuracy - permuted_accuracy),
            })
    return pd.DataFrame(records)


def run_dev_experiment_against(experiment_id, config, comparison_id, df=train_raw):
    comparison_key = f"{comparison_id}::development"
    if comparison_key not in RESEARCH_RESULTS_CACHE:
        raise RuntimeError(f"Development result missing for comparison model {comparison_id}")
    parent = RESEARCH_RESULTS_CACHE[comparison_key]
    baseline = ensure_development_baseline(df)
    result = _run_fold_set(experiment_id, config, df, get_development_folds(), "development")
    parent_comparison = summarize_candidate_vs_baseline(result, parent, df)
    baseline_comparison = summarize_candidate_vs_baseline(result, baseline, df)
    result["comparison"] = parent_comparison
    result["comparison_vs_A0"] = baseline_comparison
    RESEARCH_RESULTS_CACHE[f"{experiment_id}::development"] = result
    _upsert_research_log(
        result,
        baseline_comparison,
        decision="PASS_DEV" if parent_comparison["gate_passed"] else "REJECT_DEV",
        gate_override=parent_comparison["gate_passed"],
    )
    return result


def build_pruning_registry():
    if len(SELECTED_PRUNING_FAMILIES) > 3:
        raise ValueError("At most three feature families may be tested for removal")
    frozen = get_frozen_representation_config()
    registry = {}
    for position, family in enumerate(SELECTED_PRUNING_FAMILIES, start=1):
        registry[f"F{position}"] = {
            **deepcopy(frozen),
            "experiment_id": f"F{position}",
            "hypothesis": f"Removing weak family '{family}' improves stability",
            "drop_families": tuple(set(frozen.get("drop_families", ())) | {family}),
        }
    return registry


if RUN_FEATURE_PRUNING:
    F_REGISTRY = build_pruning_registry()
    F_RESULTS = {
        experiment_id: run_dev_experiment_against(
            experiment_id,
            config,
            FROZEN_REPRESENTATION_EXPERIMENT_ID,
        )
        for experiment_id, config in F_REGISTRY.items()
    }
else:
    F_REGISTRY = {}
    F_RESULTS = {}
    print("Experiment F is prepared; RUN_FEATURE_PRUNING=False.")




## 12. Experiment G — Compact CatBoost tuning

**What.** Search at most ten fixed CatBoost configurations after representation choices are frozen.

**How.** Six depth/schedule combinations form stage one; four regularization or quantization variants are allowed only around one human-selected stage-one result.

**Why.** Capacity and regularization may matter, but a large optimizer would meta-overfit three development folds.

**Connection to the project.** Tuning is restricted to improving the same future-sign classifier after its data representation is already justified.



In [ ]:
SELECTED_TUNING_BASE_ID = None


def build_catboost_tuning_stage1_registry():
    frozen = get_frozen_representation_config()
    registry = {}
    position = 0
    for depth, (iterations, learning_rate) in itertools.product(
        (4, 5, 6), ((500, 0.03), (800, 0.02))
    ):
        params = deepcopy(frozen["model_params"])
        params.update({
            "depth": depth,
            "iterations": iterations,
            "learning_rate": learning_rate,
        })
        experiment_id = f"G{position}"
        registry[experiment_id] = {
            **deepcopy(frozen),
            "experiment_id": experiment_id,
            "hypothesis": (
                f"Depth {depth} with {iterations} trees at learning rate {learning_rate} "
                "better matches the weak signal"
            ),
            "model_params": params,
        }
        position += 1
    assert len(registry) == 6
    return registry


def build_catboost_tuning_stage2_registry(stage1_registry):
    if SELECTED_TUNING_BASE_ID is None:
        return {}
    if SELECTED_TUNING_BASE_ID not in stage1_registry:
        raise KeyError("SELECTED_TUNING_BASE_ID must be one of G0–G5")
    if f"{SELECTED_TUNING_BASE_ID}::development" not in RESEARCH_RESULTS_CACHE:
        raise RuntimeError("Run and review stage one before selecting its base")
    base = stage1_registry[SELECTED_TUNING_BASE_ID]
    variants = [
        ("G6", {"l2_leaf_reg": 10.0}, "Stronger L2 leaf regularization improves transport"),
        ("G7", {"random_strength": 0.5}, "Lower split randomness better preserves weak signal"),
        ("G8", {"random_strength": 2.0}, "Higher split randomness reduces overfit"),
        ("G9", {"border_count": 128}, "Coarser numeric quantization reduces variance"),
    ]
    registry = {}
    for experiment_id, update, hypothesis in variants:
        params = deepcopy(base["model_params"])
        params.update(update)
        registry[experiment_id] = {
            **deepcopy(base),
            "experiment_id": experiment_id,
            "hypothesis": hypothesis,
            "model_params": params,
        }
    assert len(stage1_registry) + len(registry) <= 10
    return registry


if RUN_CATBOOST_TUNING:
    G_STAGE1_REGISTRY = build_catboost_tuning_stage1_registry()
    G_STAGE1_RESULTS = {
        experiment_id: run_dev_experiment_against(
            experiment_id,
            config,
            FROZEN_REPRESENTATION_EXPERIMENT_ID,
        )
        for experiment_id, config in G_STAGE1_REGISTRY.items()
    }
    G_STAGE2_REGISTRY = build_catboost_tuning_stage2_registry(G_STAGE1_REGISTRY)
    G_STAGE2_RESULTS = {
        experiment_id: run_dev_experiment_against(
            experiment_id,
            config,
            SELECTED_TUNING_BASE_ID,
        )
        for experiment_id, config in G_STAGE2_REGISTRY.items()
    }
else:
    G_STAGE1_REGISTRY = {}
    G_STAGE2_REGISTRY = {}
    G_STAGE1_RESULTS = {}
    G_STAGE2_RESULTS = {}
    print("Experiment G is prepared; RUN_CATBOOST_TUNING=False.")




## 13. Experiment H — Optional target-magnitude weighting

**What.** Test exactly two mild training-weight schemes without deleting near-zero future returns.

**How.** Cutoffs are learned inside each training fold; validation accuracy stays unweighted and allocation encoding remains equal-weight to isolate the loss change.

**Why.** Near-zero signs may be ambiguous, but the official metric still gives every row equal importance, so weighting can also hurt.

**Connection to the project.** The test asks whether emphasizing clearer return moves helps the classifier predict all allocation signs more accurately.



In [ ]:
def build_magnitude_weighting_registry():
    frozen = get_frozen_representation_config()
    return {
        "H1": {
            **deepcopy(frozen),
            "experiment_id": "H1",
            "hypothesis": "Half-weighting the smallest target-magnitude decile reduces label ambiguity",
            "training_policy": "magnitude_bottom_decile_half",
            "encoder_follows_training_weights": False,
        },
        "H2": {
            **deepcopy(frozen),
            "experiment_id": "H2",
            "hypothesis": "Smooth clipped target-magnitude weights allocate capacity to clearer signs",
            "training_policy": "magnitude_smooth",
            "encoder_follows_training_weights": False,
        },
    }


if RUN_MAGNITUDE_WEIGHTING:
    H_REGISTRY = build_magnitude_weighting_registry()
    H_RESULTS = {
        experiment_id: run_dev_experiment_against(
            experiment_id,
            config,
            FROZEN_REPRESENTATION_EXPERIMENT_ID,
        )
        for experiment_id, config in H_REGISTRY.items()
    }
else:
    H_REGISTRY = {}
    H_RESULTS = {}
    print("Experiment H is prepared; RUN_MAGNITUDE_WEIGHTING=False.")




## 14. Candidate selection and locked confirmation

**What.** Expose fold 3 to exactly one explicitly selected candidate that already passed the development gate.

**How.** `SELECTED_CONFIRMATION_EXPERIMENT_ID` defaults to `None`; the helper checks the cached gate before running candidate and baseline on fold 3.

**Why.** Automatic confirmation of every candidate would turn the locked fold into another tuning set.

**Connection to the project.** Confirmation tests whether a proposed allocation-sign improvement survives the latest unseen training period.



In [ ]:
SELECTED_CONFIRMATION_EXPERIMENT_ID = None
FINAL_BOOTSTRAP_MIN_LOWER_BOUND = -0.0002


def _combine_stage_results(development, confirmation, df):
    if np.any(development["covered_mask"] & confirmation["covered_mask"]):
        raise ValueError("Development and confirmation coverage overlap")
    probability = development["oof_probability"].copy()
    fold_ids = development["fold_ids"].copy()
    probability[confirmation["covered_mask"]] = confirmation["oof_probability"][confirmation["covered_mask"]]
    fold_ids[confirmation["covered_mask"]] = confirmation["fold_ids"][confirmation["covered_mask"]]
    covered = np.isfinite(probability) & (fold_ids >= 0)
    binary = np.full(len(df), -1, dtype=int)
    binary[covered] = (probability[covered] >= 0.5).astype(int)
    y = df["binary_target"].to_numpy(dtype=int)
    return {
        **deepcopy(development),
        "stage": "final_four_fold",
        "oof_probability": probability,
        "oof_binary": binary,
        "fold_ids": fold_ids,
        "covered_mask": covered,
        "fold_metrics": pd.concat(
            [development["fold_metrics"], confirmation["fold_metrics"]], ignore_index=True
        ).sort_values("fold").reset_index(drop=True),
        "pooled_accuracy": float(accuracy_score(y[covered], binary[covered])),
        "pooled_log_loss": float(log_loss(y[covered], probability[covered], labels=[0, 1])),
        "predicted_positive_rate": float(binary[covered].mean()),
        "runtime_seconds": development["runtime_seconds"] + confirmation["runtime_seconds"],
        "fitted_folds": development["fitted_folds"] + confirmation["fitted_folds"],
    }


def _candidate_development_result(experiment_id):
    key = f"{experiment_id}::development"
    if key not in RESEARCH_RESULTS_CACHE:
        raise KeyError(f"No development result for {experiment_id}")
    result = RESEARCH_RESULTS_CACHE[key]
    if experiment_id == "A0" or not result.get("comparison", {}).get("gate_passed", False):
        raise PermissionError("Only a non-baseline candidate that passed development may use fold 3")
    return result


def ensure_confirmation_baseline(df=train_raw):
    key = "A0::confirmation"
    if key not in RESEARCH_RESULTS_CACHE:
        result = _run_fold_set(
            "A0",
            RESEARCH_BASELINE_CONFIG,
            df,
            [deepcopy(PURGED_FOLDS[CONFIRMATION_FOLD_ID])],
            "confirmation",
        )
        RESEARCH_RESULTS_CACHE[key] = result
    return RESEARCH_RESULTS_CACHE[key]


def run_confirmation_experiment(experiment_id, df=train_raw):
    if experiment_id != SELECTED_CONFIRMATION_EXPERIMENT_ID:
        raise PermissionError("Experiment does not match the explicit confirmation selection")
    development = _candidate_development_result(experiment_id)
    baseline_development = ensure_development_baseline(df)
    baseline_confirmation = ensure_confirmation_baseline(df)
    confirmation = _run_fold_set(
        experiment_id,
        development["config"],
        df,
        [deepcopy(PURGED_FOLDS[CONFIRMATION_FOLD_ID])],
        "confirmation",
    )
    confirmation_comparison = summarize_candidate_vs_baseline(
        confirmation, baseline_confirmation, df
    )
    candidate_final = _combine_stage_results(development, confirmation, df)
    baseline_final = _combine_stage_results(baseline_development, baseline_confirmation, df)
    final_comparison = summarize_candidate_vs_baseline(candidate_final, baseline_final, df)
    fold_deltas = final_comparison["fold_deltas"]
    final_passed = bool(
        confirmation_comparison["delta_vs_baseline"] > CONFIRMATION_MIN_DELTA
        and final_comparison["delta_vs_baseline"] >= FINAL_MIN_POOLED_GAIN
        and int((fold_deltas["delta"] > 0).sum()) >= FINAL_MIN_POSITIVE_FOLDS
        and final_comparison["paired_timestamp_bootstrap"]["confidence_interval_95"][0]
        >= FINAL_BOOTSTRAP_MIN_LOWER_BOUND
    )
    candidate_final["comparison"] = final_comparison
    candidate_final["confirmation_comparison"] = confirmation_comparison
    candidate_final["final_passed"] = final_passed
    RESEARCH_RESULTS_CACHE[f"{experiment_id}::confirmation"] = confirmation
    RESEARCH_RESULTS_CACHE[f"{experiment_id}::final"] = candidate_final
    if final_passed:
        CONFIRMED_EXPERIMENT_CONFIGS[experiment_id] = deepcopy(development["config"])
    _upsert_research_log(
        confirmation,
        confirmation_comparison,
        decision="KEEP_CONFIRMED" if final_passed else "REJECT_CONFIRMATION",
    )
    return candidate_final


if RUN_CONFIRMATION:
    if SELECTED_CONFIRMATION_EXPERIMENT_ID is None:
        raise ValueError("Set SELECTED_CONFIRMATION_EXPERIMENT_ID after reviewing development results")
    CONFIRMATION_RESULT = run_confirmation_experiment(SELECTED_CONFIRMATION_EXPERIMENT_ID)
else:
    CONFIRMATION_RESULT = None
    print("Locked confirmation is protected; RUN_CONFIRMATION=False.")

display(RESEARCH_EXPERIMENT_LOG)




## 15. Final chronological threshold calibration

**What.** Compare threshold 0.5, ordinary nested selection, strictly earlier-fold selection, and 50% shrinkage toward 0.5 for one confirmed model.

**How.** Every reported prediction uses a threshold learned outside its evaluated fold; the first chronological fold uses 0.5.

**Why.** A small threshold gain is credible only when it survives time order and is not fitted to the same labels used for final reporting.

**Connection to the project.** Calibration controls the positive/negative decision boundary after the allocation-sign model itself is frozen.



In [ ]:
def _best_threshold(y, probability, thresholds):
    accuracies = np.array([
        accuracy_score(y, probability >= threshold) for threshold in thresholds
    ], dtype=float)
    best = accuracies.max()
    return float(np.median(thresholds[np.isclose(accuracies, best)])), float(best)


def nested_threshold_predictions(y, probability, fold_ids, thresholds, mode):
    y = np.asarray(y, dtype=int)
    probability = np.asarray(probability, dtype=float)
    fold_ids = np.asarray(fold_ids, dtype=int)
    valid = np.isfinite(probability) & (fold_ids >= 0)
    unique_folds = np.sort(np.unique(fold_ids[valid]))
    predictions = np.full(len(y), -1, dtype=int)
    shrunk_predictions = np.full(len(y), -1, dtype=int)
    records = []
    for position, fold_id in enumerate(unique_folds):
        evaluation = valid & (fold_ids == fold_id)
        if mode == "global_nested":
            selection = valid & (fold_ids != fold_id)
        elif mode == "chronological":
            earlier = unique_folds[:position]
            selection = valid & np.isin(fold_ids, earlier)
        else:
            raise ValueError("mode must be global_nested or chronological")
        if selection.any():
            threshold, _ = _best_threshold(y[selection], probability[selection], thresholds)
        else:
            threshold = 0.5
        shrunk = 0.5 + 0.5 * (threshold - 0.5)
        predictions[evaluation] = (probability[evaluation] >= threshold).astype(int)
        shrunk_predictions[evaluation] = (probability[evaluation] >= shrunk).astype(int)
        records.append({
            "fold": int(fold_id),
            "selected_threshold": float(threshold),
            "shrunk_threshold": float(shrunk),
            "accuracy": float(accuracy_score(y[evaluation], predictions[evaluation])),
            "shrunk_accuracy": float(accuracy_score(y[evaluation], shrunk_predictions[evaluation])),
            "default_accuracy": float(
                accuracy_score(y[evaluation], probability[evaluation] >= 0.5)
            ),
            "n_selection": int(selection.sum()),
            "n_evaluation": int(evaluation.sum()),
        })
    return predictions, shrunk_predictions, pd.DataFrame(records), valid


def calibrate_confirmed_candidate(experiment_id, df=train_raw):
    key = f"{experiment_id}::final"
    if key not in RESEARCH_RESULTS_CACHE:
        raise KeyError("Run locked confirmation before threshold calibration")
    result = RESEARCH_RESULTS_CACHE[key]
    if not result.get("final_passed", False):
        raise PermissionError("Threshold optimization is restricted to a confirmed model")
    y = df["binary_target"].to_numpy(dtype=int)
    probability = result["oof_probability"]
    fold_ids = result["fold_ids"]
    thresholds = np.linspace(0.48, 0.52, 81)
    global_pred, global_shrunk, global_folds, valid = nested_threshold_predictions(
        y, probability, fold_ids, thresholds, "global_nested"
    )
    chrono_pred, chrono_shrunk, chrono_folds, chrono_valid = nested_threshold_predictions(
        y, probability, fold_ids, thresholds, "chronological"
    )
    assert np.array_equal(valid, chrono_valid)
    default_prediction = (probability[valid] >= 0.5).astype(int)
    current_reference = (probability[valid] >= 0.492).astype(int)
    final_global_threshold, _ = _best_threshold(y[valid], probability[valid], thresholds)
    summary = pd.DataFrame([
        {"method": "default_0.5", "nested_accuracy": accuracy_score(y[valid], default_prediction)},
        {"method": "current_reference_0.492", "nested_accuracy": accuracy_score(y[valid], current_reference)},
        {"method": "global_nested", "nested_accuracy": accuracy_score(y[valid], global_pred[valid])},
        {"method": "global_nested_shrunk", "nested_accuracy": accuracy_score(y[valid], global_shrunk[valid])},
        {"method": "chronological", "nested_accuracy": accuracy_score(y[valid], chrono_pred[valid])},
        {"method": "chronological_shrunk", "nested_accuracy": accuracy_score(y[valid], chrono_shrunk[valid])},
    ])
    summary_lookup = summary.set_index("method")["nested_accuracy"]
    chronological_method = (
        "chronological"
        if summary_lookup["chronological"] >= summary_lookup["chronological_shrunk"]
        else "chronological_shrunk"
    )
    fold_accuracy_column = "accuracy" if chronological_method == "chronological" else "shrunk_accuracy"
    fold_threshold_column = (
        "selected_threshold" if chronological_method == "chronological" else "shrunk_threshold"
    )
    chronological_gain = float(
        summary_lookup[chronological_method] - summary_lookup["default_0.5"]
    )
    nonnegative_folds = int(
        (chrono_folds[fold_accuracy_column] >= chrono_folds["default_accuracy"]).sum()
    )
    threshold_range = float(
        chrono_folds[fold_threshold_column].max() - chrono_folds[fold_threshold_column].min()
    )
    use_non_default = bool(
        chronological_gain > 0
        and nonnegative_folds >= 3
        and threshold_range <= 0.02
    )
    recommended_threshold = (
        final_global_threshold
        if use_non_default and chronological_method == "chronological"
        else 0.5 + 0.5 * (final_global_threshold - 0.5)
        if use_non_default
        else 0.5
    )
    return {
        "experiment_id": experiment_id,
        "summary": summary,
        "global_fold_thresholds": global_folds,
        "chronological_fold_thresholds": chrono_folds,
        "production_threshold_candidate": final_global_threshold,
        "production_threshold_shrunk": 0.5 + 0.5 * (final_global_threshold - 0.5),
        "chronological_method_selected": chronological_method,
        "chronological_gain_vs_0.5": chronological_gain,
        "nonnegative_chronological_folds": nonnegative_folds,
        "chronological_threshold_range": threshold_range,
        "use_non_default_threshold": use_non_default,
        "recommended_threshold": float(recommended_threshold),
    }


if RUN_FINAL_THRESHOLD_CALIBRATION:
    if SELECTED_CONFIRMATION_EXPERIMENT_ID is None:
        raise ValueError("A confirmed experiment ID is required")
    FINAL_THRESHOLD_RESULT = calibrate_confirmed_candidate(
        SELECTED_CONFIRMATION_EXPERIMENT_ID
    )
    display(FINAL_THRESHOLD_RESULT["summary"])
else:
    FINAL_THRESHOLD_RESULT = None
    print("Final threshold calibration is prepared; RUN_FINAL_THRESHOLD_CALIBRATION=False.")




## 16. Final research summary

**What.** Summarize the baseline, selected representation, confirmed result, parameters, and final threshold without generating a submission.

**How.** The table reads only the central cache, log, and explicitly selected IDs; empty fields remain pending while execution flags are off.

**Why.** A compact hand-off prevents exploratory state from being mistaken for a production decision.

**Connection to the project.** If a candidate passes confirmation, its frozen configuration can be transferred to production for one new leaderboard submission.



In [ ]:
def build_final_research_summary():
    selected_id = SELECTED_CONFIRMATION_EXPERIMENT_ID
    final_result = RESEARCH_RESULTS_CACHE.get(f"{selected_id}::final") if selected_id else None
    threshold_result = FINAL_THRESHOLD_RESULT
    return pd.DataFrame([{
        "baseline": "A0 / P2_catboost_binary_purged",
        "winning_data_representation": (
            None if final_result is None else {
                "numeric_mode": final_result["config"]["numeric_mode"],
                "encoding": final_result["config"]["encoding"],
            }
        ),
        "winning_feature_additions": (
            None if final_result is None else final_result["config"].get("extra_blocks", ())
        ),
        "winning_catboost_parameters": (
            None if final_result is None else final_result["config"]["model_params"]
        ),
        "confirmation_passed": (
            None if final_result is None else final_result.get("final_passed")
        ),
        "final_threshold": (
            None if threshold_result is None else threshold_result["recommended_threshold"]
        ),
        "expected_next_action": (
            "If the final candidate passes confirmation, transfer its configuration "
            "to the production notebook and make one new submission."
        ),
    }])


FINAL_RESEARCH_SUMMARY = build_final_research_summary()
display(FINAL_RESEARCH_SUMMARY)
